# Another answer format from the same saved execution

Complete the matching base case in notebook 01 first:

| Variant file | Required base case | New answer |
| --- | --- | --- |
| [msft_fundamentals_plain.json](cases/msft_fundamentals_plain.json) | msft_fundamentals | Plain text only |
| [sp500_momentum_json.json](cases/sp500_momentum_json.json) | sp500_momentum_screener | JSON only |

This notebook reads the original operation, supplies its exact study/execution
identities, and asks for another answer on the same thread. It does not quietly
launch the prerequisite study. A follow-up creates a distinct saved answer while
the original operation retains its own pinned revision.

The instruction forbids new discovery/execution, and the checks require the same
primary execution. A live tool-trace review is still needed to prove that no extra
job was attempted. These output variants have only been prepared offline so far.

## 1. Connect

`connect()` prepares the normal SDK client from the notebook environment.
`check_services()` checks service health; neither starts research.

In [ ]:
from pathlib import Path
from IPython.display import JSON, display
from epoch_sdk.research import ResearchCompleted
from notebook_support import connect, check_services, operation_name
from case_support import case_info, load_case, read_case_tables, check_case_answer, check_variant_source

epoch = connect()
await check_services()

## 2. Choose the output variant and read its completed source

Keep `BASE_ATTEMPT` equal to the `ATTEMPT` used in notebook 01. Reading this
operation starts no agent or job. If the base case is missing or incomplete,
finish it in notebook 01 before proceeding.

In [ ]:
# Change this to cases/sp500_momentum_json.json for the JSON shortlist variant.
CASE_FILE = Path("cases/msft_fundamentals_plain.json")
BASE_ATTEMPT = "01"
VARIANT_ATTEMPT = "01"  # Change this only for an intentionally new follow-up.
info = case_info(CASE_FILE)
assert info["kind"] == "output_variant"
SOURCE_RESEARCH_ID = operation_name(info["source_case"] + "-" + BASE_ATTEMPT + "-research")
RESEARCH_ID = operation_name(info["id"] + "-" + BASE_ATTEMPT + "-" + VARIANT_ATTEMPT + "-research")

# Fetch the original, pinned answer. This is a read, not a re-submission.
source = await epoch.research_status(operation_id=SOURCE_RESEARCH_ID)
assert isinstance(source, ResearchCompleted), "Complete the base case in notebook 01 first."
print("Source operation:", SOURCE_RESEARCH_ID)
print("Source execution:", source.response.primary_execution_id)
print("Follow-up operation:", RESEARCH_ID)

## 3. Inspect the follow-up request

The local loader adds the exact saved source IDs to the natural-language context.
It keeps the normal SDK research-request shape. No vendor credentials or SQL are
needed. The following research call also attaches to the source thread.

In [ ]:
# The loader rejects a variant without the matching completed source outputs.
request = load_case(CASE_FILE, source=source)
display(JSON(request.to_wire(), expanded=False))

## 4. Publish the new format from saved evidence

One session call shows live activity, handles any explicit questions here and
returns the saved answer. This is a new model turn over the chosen execution;
it must not refresh market data or execute a new study.


In [ ]:
# Reuse the source conversation, with a new answer operation ID.
# Progress, questions and the published answer are rendered inside this cell.
result = await epoch.run(
    request,
    thread_id=source.response.thread_id,
    operation_id=RESEARCH_ID,
)
assert isinstance(result, ResearchCompleted), "Continue the session before checking the answer."


## 5. Check source reuse and the original answer pin

The JSON screener variant must match every value and the order of the original
table, including coverage metadata. Plain-text factual fidelity needs human
review. Both variants must reference the original execution and a new answer ID.

In [ ]:
# Read the original tables for comparison; this starts no graph or engine job.
source_tables = await read_case_tables(epoch, source)
variant_tables = await read_case_tables(epoch, result)
check_case_answer(CASE_FILE, request, result, variant_tables)
check_variant_source(CASE_FILE, source, result, source_tables)

# A new answer must not change what the original research operation returns.
original = await epoch.research_status(operation_id=SOURCE_RESEARCH_ID)
assert isinstance(original, ResearchCompleted)
assert original.response == source.response
print("Original answer revision is still:", original.response.version)

## 6. Close the client

Keep both operation IDs to reconnect later. The original study, execution and
answers remain on the services. The same case JSON can be supplied to MCP's
`epoch_research`; pass the original thread for an output-only follow-up.

In [ ]:
# Close only this Python client's connections.
await epoch.aclose()